# Transfer Learning

In [1]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras import Sequential

In [5]:
train = "FruitinAmazon/train"
test = "FruitinAmazon/test"

In [6]:
os.listdir("FruitinAmazon/test")

['acai', 'cupuacu', 'graviola', 'guarana', 'pupunha', 'tucuma']

In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    train,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    train,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 30 images belonging to 6 classes.


In [8]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.1389 - loss: 2.0921 - val_accuracy: 0.3333 - val_loss: 1.3873
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 449ms/step - accuracy: 0.5139 - loss: 1.2718 - val_accuracy: 0.7778 - val_loss: 0.9740
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 837ms/step - accuracy: 0.7361 - loss: 0.7380 - val_accuracy: 0.7778 - val_loss: 0.7130
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 861ms/step - accuracy: 0.8472 - loss: 0.5085 - val_accuracy: 0.7778 - val_loss: 0.6182
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.8889 - loss: 0.3841 - val_accuracy: 0.8333 - val_loss: 0.5708
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 864ms/step - accuracy: 0.9306 - loss: 0.2387 - val_accuracy: 0.8333 - val_loss: 0.4660
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 837ms/step - accuracy: 0.9583 - loss: 0.2396 - val_accuracy: 0.8333 - val_loss: 0.3954
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 856ms/step - accuracy: 0.9583 - loss: 0.1479 - val_accuracy: 0.8333 - val_loss: 0.35

In [10]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.9333 - loss: 0.3338
Test Accuracy: 0.9333333373069763


In [11]:
predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
              precision    recall  f1-score   support

        acai       1.00      1.00      1.00         5
     cupuacu       1.00      1.00      1.00         5
    graviola       1.00      1.00      1.00         5
     guarana       1.00      1.00      1.00         5
     pupunha       0.71      1.00      0.83         5
      tucuma       1.00      0.60      0.75         5

    accuracy                           0.93        30
   macro avg       0.95      0.93      0.93        30
weighted avg       0.95      0.93      0.93        30



In [ ]:
img_path = "FruitinAmazon/train/pupunha/images (1).jpeg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)
pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
Prediction: pupunha


In [17]:
scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

C:\Users\LEGION\OneDrive\Desktop\Sem6\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.1389 - loss: 20.0291 - val_accuracy: 0.1667 - val_loss: 11.3577
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.3056 - loss: 6.0208 - val_accuracy: 0.3333 - val_loss: 2.0907
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.3611 - loss: 1.7921 - val_accuracy: 0.2222 - val_loss: 1.6536
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 988ms/step - accuracy: 0.3889 - loss: 1.3974 - val_accuracy: 0.3889 - val_loss: 1.5252
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 979ms/step - accuracy: 0.6944 - loss: 1.0856 - val_accuracy: 0.2778 - val_loss: 1.5516
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 970ms/step - accuracy: 0.8056 - loss: 0.7527 - val_accuracy: 0.3889 - val_loss: 1.4805
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.7500 - loss: 0.6097 - val_accuracy: 0.3889 - val_loss: 1.4656
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.9444 - loss: 0.2899 - val_accuracy: 0.6111 - val_loss: 1.2122
Epoch

In [18]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step
              precision    recall  f1-score   support

        acai       0.56      1.00      0.71         5
     cupuacu       0.50      0.20      0.29         5
    graviola       0.56      1.00      0.71         5
     guarana       0.67      0.40      0.50         5
     pupunha       1.00      0.40      0.57         5
      tucuma       0.40      0.40      0.40         5

    accuracy                           0.57        30
   macro avg       0.61      0.57      0.53        30
weighted avg       0.61      0.57      0.53        30



In [19]:
img_path = "FruitinAmazon/train/pupunha/images (1).jpeg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
Prediction: tucuma
